# **<span style="color:blue">Notebook 10 - Prof. Dr. Humberto Elias Garcia Lopes - Acesso a Bases de Dados Públicas de Economia</span>**

Neste Notebook, você aprenderá a usar o Python para buscar dados econômicos diretamente nos sites de instituições públicas brasileiras, sem precisar baixar planilhas manualmente. Trabalharemos com três fontes: o **Banco Central do Brasil (BCB)**, o **Instituto Brasileiro de Geografia e Estatística (IBGE)** e o **Instituto de Pesquisa Econômica Aplicada (Ipea)**.

Você verá que os dados chegam ao Python na forma de **listas** e **dicionários**, estruturas que já estudamos nos Notebooks anteriores. Ao final, conheceremos a biblioteca **pandas**, que organiza esses dados em tabelas, e faremos um gráfico simples.

**Pré-requisitos:** listas, dicionários, laços **for**, condicionais **if** e criação de funções com **def**.

**Atenção:** para executar este Notebook, o computador precisa estar conectado à internet.

### **<span style="color:darkgreen">1) O que são bases de dados públicas?</span>**

Diversos órgãos públicos divulgam gratuitamente as estatísticas que produzem. Para um economista, as mais usadas são:

a) **Banco Central do Brasil**: taxa Selic, inflação (IPCA), câmbio, crédito, entre outras. Os dados ficam no **Sistema Gerenciador de Séries Temporais (SGS)**, em que cada série tem um número de identificação (código).

b) **IBGE**: PIB, população, desemprego, inflação, produção industrial. Os dados ficam no **SIDRA** (Sistema IBGE de Recuperação Automática), organizados em tabelas numeradas.

c) **Ipeadata**: base mantida pelo Ipea que reúne séries de várias fontes, como salário mínimo, renda e indicadores regionais.

Essas três instituições oferecem uma **API** (do inglês *Application Programming Interface*). Em termos simples, uma API é um endereço de internet que, em vez de devolver uma página para ser lida por pessoas, devolve dados prontos para serem lidos por um programa.

> **1.1) Como funciona uma API?**

O processo tem três etapas:

a) Montamos um **endereço (URL)** que descreve os dados desejados. Por exemplo, o endereço abaixo pede ao Banco Central os 3 últimos valores do IPCA mensal (série 433):

```
https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados/ultimos/3?formato=json
```

b) O Python **envia um pedido** (requisição) para esse endereço.

c) O servidor **responde** com os dados em um formato chamado **JSON**. O JSON é muito parecido com as listas e os dicionários do Python. A resposta do exemplo acima tem esta forma:

```
[{"data": "01/07/2025", "valor": "0.26"},
 {"data": "01/08/2025", "valor": "-0.11"},
 {"data": "01/09/2025", "valor": "0.48"}]
```

Ou seja, uma **lista de dicionários**, em que cada dicionário guarda a data e o valor de um mês. Os valores mostrados aqui servem apenas para ilustrar o formato; ao executar o código, você receberá os dados mais recentes.

**Experimente:** copie o endereço do item a) e cole-o no seu navegador. Você verá o texto em JSON na tela.

### **<span style="color:darkgreen">2) As bibliotecas que vamos usar</span>**

Uma **biblioteca** é um conjunto de funções prontas, escritas por outros programadores, que podemos importar para o nosso programa com o comando **import**. Usaremos três:

a) **requests**: envia pedidos para endereços de internet e recebe as respostas.

b) **pandas**: organiza dados em tabelas, parecidas com as de uma planilha.

c) **matplotlib**: faz gráficos.

No Google Colab e na distribuição Anaconda, as três já vêm instaladas. Se aparecer o erro *ModuleNotFoundError*, execute em uma célula o comando `!pip install requests pandas matplotlib` e depois rode novamente a célula de importação.

In [ ]:
# Código para importar as bibliotecas

import requests                  # Envia pedidos para a internet
import pandas as pd              # "pd" é um apelido para não precisarmos escrever "pandas" toda vez
import matplotlib.pyplot as plt  # "plt" é o apelido usado para os gráficos

### **<span style="color:darkgreen">3) Primeiro exemplo: o IPCA no Banco Central</span>**

O **IPCA** (Índice Nacional de Preços ao Consumidor Amplo) é o índice oficial de inflação do Brasil. No SGS do Banco Central, a variação mensal do IPCA corresponde à série de código **433**.

Vamos pedir os valores mensais de janeiro a dezembro de 2024. Para isso, acrescentamos ao endereço as datas inicial e final, sempre no formato dia/mês/ano.

> **3.1) Enviando o pedido**

In [ ]:
# Código para buscar a variação mensal do IPCA em 2024

url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01/01/2024&dataFinal=31/12/2024"

resposta = requests.get(url, timeout=30)  # Envia o pedido e espera, no máximo, 30 segundos pela resposta

print(resposta.status_code)  # Mostra o código de situação da resposta

O **status_code** informa se o pedido deu certo. Os códigos mais comuns são:

a) **200**: tudo certo, os dados foram enviados.

b) **404**: o endereço não existe (verifique se digitou a URL corretamente).

c) **500** ou **503**: problema no servidor da instituição. Nesse caso, aguarde alguns minutos e tente de novo.

> **3.2) Lendo os dados recebidos**

O método **.json()** transforma o texto recebido em listas e dicionários do Python.

In [ ]:
# Código para converter a resposta em uma lista de dicionários

dados = resposta.json()

print(type(dados))  # Confirma que recebemos uma lista
print(len(dados))   # Quantidade de meses recebidos
print(dados[0])     # Primeiro elemento da lista: um dicionário

Como cada elemento da lista é um dicionário, podemos acessar a data e o valor pelas chaves **"data"** e **"valor"**, exatamente como fizemos no Notebook sobre dicionários.

In [ ]:
# Código para exibir os valores mês a mês

for i in range(len(dados)):
    mes = dados[i]["data"]
    valor = dados[i]["valor"]
    print(f"{mes}: {valor}%")

> **3.3) Atenção ao tipo dos valores**

Observe a resposta mostrada no item 1.1: os valores aparecem entre aspas, como **"0.26"**. Isso significa que o Banco Central envia os números como **texto (str)**. Se tentarmos fazer contas com eles, teremos um erro.

In [ ]:
# Código para mostrar o erro de somar texto com número. Isso causará um erro!

print(dados[0]["valor"] + 1)

A solução é converter o texto em número com a função **float()**, que já conhecemos.

In [ ]:
# Código para converter o valor de texto para número decimal

valor_janeiro = float(dados[0]["valor"])

print(valor_janeiro + 1)

> **3.4) Calculando a inflação acumulada no ano**

A inflação acumulada **não** é a soma das taxas mensais. Ela é obtida pela multiplicação dos fatores de cada mês:

$$\text{Acumulado} = (1 + i_1) \times (1 + i_2) \times \dots \times (1 + i_{12}) - 1$$

em que $i_1, i_2, \dots$ são as taxas mensais na forma decimal (por exemplo, 0,26% = 0,0026). Vamos usar um **acumulador** dentro de um laço **for**.

In [ ]:
# Código para calcular a inflação acumulada em 2024

fator = 1  # Acumulador da multiplicação. Começa em 1, e não em 0!

for i in range(len(dados)):
    taxa = float(dados[i]["valor"]) / 100  # Converte para número e para a forma decimal
    fator = fator * (1 + taxa)

acumulado = (fator - 1) * 100

print(f"Inflação acumulada (IPCA) em 2024: {acumulado:.2f}%")

Para conferir o resultado, compare-o com a inflação oficial de 2024 divulgada pelo IBGE. Pequenas diferenças na segunda casa decimal podem ocorrer por arredondamento das taxas mensais.

### **<span style="color:darkgreen">4) Criando uma função para buscar qualquer série do Banco Central</span>**

O SGS tem milhares de séries, e a única parte do endereço que muda de uma para outra é o **código**. Por isso, convém criar uma função que recebe o código e as datas e devolve a lista de dados. Alguns códigos úteis:

| Código | Série | Periodicidade |
|---|---|---|
| 433 | IPCA, variação mensal (%) | mensal |
| 432 | Meta da taxa Selic definida pelo Copom (% ao ano) | diária |
| 4390 | Taxa Selic acumulada no mês (%) | mensal |
| 1 | Taxa de câmbio, dólar americano, venda (R$) | diária |
| 24369 | Taxa de desocupação, PNAD Contínua (%) | mensal |

Outros códigos podem ser pesquisados no site do SGS: https://www3.bcb.gov.br/sgspub

In [ ]:
# Código para criar uma função que busca uma série do SGS do Banco Central

def buscar_serie_bcb(codigo, data_inicial, data_final):
    url = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json&dataInicial={data_inicial}&dataFinal={data_final}"
    resposta = requests.get(url, timeout=30)
    if resposta.status_code == 200:
        return resposta.json()
    else:
        print(f"Erro ao buscar a série {codigo}. Código de situação: {resposta.status_code}")
        return []  # Devolve uma lista vazia quando algo dá errado

Agora podemos buscar a taxa de câmbio (série 1) em poucos dias, com uma única linha.

In [ ]:
# Código para buscar a cotação do dólar na primeira semana de outubro de 2025

dolar = buscar_serie_bcb(1, "01/10/2025", "07/10/2025")

for i in range(len(dolar)):
    print(f"{dolar[i]['data']}: R$ {float(dolar[i]['valor']):.4f}")

Observe que dentro da f-string usamos aspas simples em **dolar[i]['data']**, porque a f-string já foi aberta com aspas duplas.

**Atenção:** para as séries diárias, como a do câmbio, o Banco Central limita cada consulta a um período de até 10 anos. Para períodos maiores, é preciso dividir a busca em partes.

### **<span style="color:darkgreen">5) Organizando os dados em tabela com a biblioteca pandas</span>**

Percorrer listas de dicionários com **for** funciona bem, mas, quando os dados são muitos, é mais prático organizá-los em uma tabela. A biblioteca **pandas** chama essa tabela de **DataFrame**. Cada chave do dicionário vira uma coluna, e cada dicionário vira uma linha.

> **5.1) Criando um DataFrame**

In [ ]:
# Código para transformar a lista de dicionários em uma tabela (DataFrame)

selic = buscar_serie_bcb(4390, "01/01/2024", "31/12/2024")  # Selic acumulada no mês

tabela = pd.DataFrame(selic)

tabela.head()  # Mostra as 5 primeiras linhas da tabela

Alguns comandos ajudam a conhecer a tabela:

a) **tabela.head()** mostra as primeiras linhas, e **tabela.tail()** mostra as últimas.

b) **tabela.shape** informa o número de linhas e de colunas.

c) **tabela.dtypes** mostra o tipo de cada coluna. O tipo **object** (ou **str**, nas versões mais recentes do pandas) indica texto.

In [ ]:
# Código para conferir o tamanho da tabela e o tipo das colunas

print(tabela.shape)
print(tabela.dtypes)

> **5.2) Convertendo os tipos das colunas**

Assim como no item 3.3, as duas colunas chegaram como texto. No pandas, convertemos uma coluna inteira de uma só vez:

a) **pd.to_numeric()** converte texto em número.

b) **pd.to_datetime()** converte texto em data. O argumento **dayfirst=True** avisa que o dia vem antes do mês, como no padrão brasileiro.

In [ ]:
# Código para converter as colunas para número e para data

tabela["valor"] = pd.to_numeric(tabela["valor"])
tabela["data"] = pd.to_datetime(tabela["data"], dayfirst=True)

print(tabela.dtypes)

> **5.3) Resumindo e ordenando os dados**

In [ ]:
# Código para obter estatísticas da taxa Selic mensal em 2024

print(f"Média mensal: {tabela['valor'].mean():.2f}%")
print(f"Maior taxa mensal: {tabela['valor'].max():.2f}%")
print(f"Menor taxa mensal: {tabela['valor'].min():.2f}%")

In [ ]:
# Código para ordenar a tabela da maior para a menor taxa

tabela.sort_values("valor", ascending=False).head(3)

> **5.4) Fazendo um gráfico**

Com a coluna de datas já convertida, um gráfico de linha mostra a trajetória da série ao longo do tempo.

In [ ]:
# Código para fazer o gráfico da Selic mensal em 2024

plt.plot(tabela["data"], tabela["valor"], marker="o")
plt.title("Taxa Selic acumulada no mês, 2024")
plt.xlabel("Mês")
plt.ylabel("% ao mês")
plt.grid(True)
plt.show()

### **<span style="color:darkgreen">6) Segundo exemplo: dados do IBGE (SIDRA)</span>**

No SIDRA, os dados ficam em tabelas numeradas, e o endereço da API é montado em partes separadas por barras. Por exemplo:

```
https://apisidra.ibge.gov.br/values/t/6579/n3/all/v/9324/p/last%201
```

a) **t/6579**: tabela 6579 (estimativas da população residente).

b) **n3/all**: nível territorial 3 (Unidades da Federação), todas elas. Use **n1/all** para o Brasil e **n6/all** para os municípios.

c) **v/9324**: variável 9324 (população residente estimada).

d) **p/last%201**: o último período disponível. O **%20** representa um espaço em branco no endereço, de modo que lemos "last 1".

Para descobrir os números de outras tabelas e variáveis, use a página https://sidra.ibge.gov.br e o construtor de consultas em https://apisidra.ibge.gov.br

In [ ]:
# Código para buscar a população estimada de cada estado no SIDRA

url = "https://apisidra.ibge.gov.br/values/t/6579/n3/all/v/9324/p/last%201"

resposta = requests.get(url, timeout=30)
dados_ibge = resposta.json()

print(len(dados_ibge))  # 27 estados + 1 linha de cabeçalho
print(dados_ibge[0])    # A primeira linha é o cabeçalho

A resposta do SIDRA também é uma lista de dicionários, com uma particularidade: o **primeiro** dicionário é um cabeçalho, que traduz as chaves abreviadas (**"V"**, **"D1N"** etc.) para nomes legíveis (**"Valor"**, **"Unidade da Federação"** etc.).

Por isso, criamos a tabela a partir do segundo elemento (**dados_ibge[1:]**) e usamos o primeiro elemento para renomear as colunas com o método **.rename()**.

In [ ]:
# Código para criar a tabela do IBGE com nomes de colunas legíveis

populacao = pd.DataFrame(dados_ibge[1:])          # Ignora a linha de cabeçalho
populacao = populacao.rename(columns=dados_ibge[0])  # Usa o cabeçalho para renomear as colunas

populacao.head()

In [ ]:
# Código para manter apenas as colunas de interesse e converter a população em número

populacao = populacao[["Unidade da Federação", "Valor"]]
populacao["Valor"] = pd.to_numeric(populacao["Valor"])

populacao.sort_values("Valor", ascending=False).head(5)  # Os 5 estados mais populosos

In [ ]:
# Código para calcular a participação de cada estado na população do Brasil

total = populacao["Valor"].sum()
populacao["Participação (%)"] = populacao["Valor"] / total * 100

print(f"População total estimada: {total:,.0f} habitantes".replace(",", "."))  # Troca a vírgula pelo ponto, como no padrão brasileiro
populacao.sort_values("Participação (%)", ascending=False).head(5)

### **<span style="color:darkgreen">7) Terceiro exemplo: o Ipeadata</span>**

No Ipeadata, cada série também tem um código, desta vez formado por letras e números. A série do salário mínimo vigente, por exemplo, tem o código **MTE12_SALMIN12**. Os códigos aparecem na página de cada série em http://www.ipeadata.gov.br

A resposta do Ipeadata é um **dicionário**, e a lista com os dados fica guardada na chave **"value"**. Em cada registro, a data está na chave **"VALDATA"** e o valor, na chave **"VALVALOR"**.

In [ ]:
# Código para buscar a série histórica do salário mínimo no Ipeadata

url = "http://www.ipeadata.gov.br/api/odata4/ValoresSerie(SERCODIGO='MTE12_SALMIN12')"

resposta = requests.get(url, timeout=60)
dados_ipea = resposta.json()["value"]  # A lista de dados fica na chave "value"

salario = pd.DataFrame(dados_ipea)
salario = salario[["VALDATA", "VALVALOR"]]

salario.tail()  # Últimos registros

Repare que, no Ipeadata, os valores já chegam como número, mas a data vem acompanhada de hora e fuso horário (por exemplo, **2025-01-01T00:00:00-03:00**). Os 10 primeiros caracteres desse texto formam a data. Por isso, usamos o fatiamento de strings que já conhecemos: no pandas, **.str[:10]** aplica o fatiamento a todas as linhas da coluna. Depois, convertemos o resultado em data com **pd.to_datetime()**.

In [ ]:
# Código para limpar a data e fazer o gráfico do salário mínimo desde 2010

salario["VALDATA"] = pd.to_datetime(salario["VALDATA"].str[:10])  # Fica só com "AAAA-MM-DD"
salario_recente = salario[salario["VALDATA"] >= "2010-01-01"]  # Filtra a partir de 2010

plt.plot(salario_recente["VALDATA"], salario_recente["VALVALOR"])
plt.title("Salário mínimo vigente (R$)")
plt.xlabel("Ano")
plt.ylabel("R$")
plt.grid(True)
plt.show()

A linha **salario[salario["VALDATA"] >= ...]** é um **filtro**: ela mantém apenas as linhas em que a condição entre colchetes é verdadeira, como um **if** aplicado a todas as linhas da tabela de uma só vez.

### **<span style="color:darkgreen">8) Cuidados ao trabalhar com APIs</span>**

a) **Conexão:** sem internet, o comando **requests.get()** gera um erro. Verifique a conexão antes de procurar problemas no código.

b) **Instabilidade:** os servidores públicos às vezes ficam lentos ou fora do ar. Por isso usamos o argumento **timeout**, que evita que o programa fique esperando indefinidamente.

c) **Tipos:** confira sempre se os valores chegaram como texto ou como número e converta-os antes de fazer contas.

d) **Unidades:** leia a descrição da série. Uma taxa pode estar em % ao mês ou em % ao ano, e um valor pode estar em reais correntes ou em reais de um ano-base.

e) **Fonte:** em trabalhos e relatórios, informe sempre a fonte dos dados, o código da série e a data da consulta.

### **<span style="color:darkgreen">9) Exercício proposto</span>**

**Juro real no Brasil**

A taxa de juros real mostra quanto um investimento rende acima da inflação. Ela é calculada pela **equação de Fisher**:

$$\text{Juro real} = \frac{1 + \text{juro nominal}}{1 + \text{inflação}} - 1$$

com as taxas na forma decimal. Usando a função **buscar_serie_bcb()** criada no item 4, faça um programa que:

a) Busque a **Selic acumulada no mês** (série 4390) e o **IPCA mensal** (série 433) de janeiro a dezembro de 2024.

b) Crie uma função chamada **acumular(dados)** que receba a lista de dicionários devolvida pelo Banco Central e retorne a taxa acumulada no período, em porcentagem (use como modelo o item 3.4).

c) Use a função **acumular()** para calcular a Selic acumulada e o IPCA acumulado em 2024.

d) Calcule o juro real de 2024 pela equação de Fisher.

e) Exiba os três resultados com duas casas decimais, usando f-string, em um pequeno relatório como o modelo abaixo (os números são apenas ilustrativos):

```
===== JUROS E INFLAÇÃO EM 2024 =====
Selic acumulada:  10,00%
IPCA acumulado:    4,00%
Juro real:         5,77%
```

f) **Desafio:** crie um DataFrame com a Selic mensal e outro com o IPCA mensal, faça um único gráfico com as duas séries e indique, em um comentário no código, em quais meses a Selic mensal ficou abaixo do IPCA mensal.

**Dica:** o mesmo programa pode ser usado para qualquer outro ano. Basta trocar as datas. Experimente também para 2025 e compare os resultados.